**MD Shohidul Islam ** mail: mdshohidulislam25100@gmail.com

## 1. Data Loading

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/stock_data.csv')

# Display the first few rows
print('First 5 rows of the dataset:')
display(df.head())

# Display the shape of the dataset
print('\nShape of the dataset (rows, columns):')
print(df.shape)

First 5 rows of the dataset:


,Unnamed: 0,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5
0,2020-01-01,101.764052,100.160928,99.494642,99.909756,101.761266
1,2020-01-02,102.171269,99.969968,98.682973,100.640755,102.528643
2,2020-01-03,103.171258,99.575237,98.182139,100.574847,101.887811
3,2020-01-04,105.483215,99.308641,97.149381,100.925017,101.490049
4,2020-01-05,107.453175,98.188428,99.575396,101.594411,101.604283



Shape of the dataset (rows, columns):
(365, 6)


## 2. Professional Data Preprocessing

To ensure our model receives high-quality data, we perform the following professional preprocessing steps:
1. **Standardization**: Renaming columns for consistency.
2. **Temporal Alignment**: Converting date strings to formal datetime objects.
3. **Integrity Check**: Verifying the dataset for null values to prevent runtime errors.
4. **Target Labeling**: Creating a 'Target' variable by shifting stock prices to predict the *next* day.
5. **Indicator Engineering**: Calculating a 7-day Moving Average (MA7) to capture short-term momentum trends.

In [3]:
# 1. Rename column and 2. Convert to datetime
df.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])

# 3. Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

# 4. Feature Engineering: Create Target (Shifted Stock_1 price)
df['Target'] = df['Stock_1'].shift(-1)

# 5. Feature Engineering: 7-day Rolling Mean
df['Stock_1_MA7'] = df['Stock_1'].rolling(window=7).mean()

# Drop rows with NaN created by shift/rolling
df.dropna(inplace=True)

print('\nPreprocessing complete. First 5 rows after cleaning:')
display(df.head())
print(f'New shape: {df.shape}')

Missing values per column:
Date       0
Stock_1    0
Stock_2    0
Stock_3    0
Stock_4    0
Stock_5    0
dtype: int64

Preprocessing complete. First 5 rows after cleaning:


,Date,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5,Target,Stock_1_MA7
6,2020-01-07,107.413982,97.485922,97.888611,100.441100,101.006372,107.251404,104.837144
7,2020-01-08,107.251404,98.306394,96.631181,102.026929,101.791802,107.140700,105.621052
8,2020-01-09,107.140700,98.061160,96.530353,101.215305,101.755418,107.580618,106.330971
9,2020-01-10,107.580618,98.109696,95.576631,100.641981,102.097333,107.735581,106.960879
10,2020-01-11,107.735581,98.594197,94.451093,100.332314,103.002417,109.302351,107.282646


New shape: (358, 8)


## 3 & 4. Automated Pipeline & Model Selection

**Strategy:** We utilize a `Pipeline` to encapsulate the scaling and modeling process. This prevents data leakage and ensures that our test data is transformed using the exact parameters derived from the training set.

**Model Choice: Random Forest Regressor**
* **Robustness**: It is highly effective at handling non-linear patterns inherent in stock market fluctuations.
* **Stability**: By averaging multiple decision trees, it reduces variance and prevents overfitting compared to a single deep tree.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Define features (X) and target (y)
X = df.drop(columns=['Date', 'Target'])
y = df['Target']

# Split the data (using shuffle=False for time-series context)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Create the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

print('Pipeline created and data split into training and testing sets.')

Pipeline created and data split into training and testing sets.


## 5, 6 & 7. Model Optimization (Training & Tuning)

We don't just train; we optimize. We use **Cross-Validation** to ensure the model's performance isn't a fluke of the data split, and **Grid Search** to mathematically find the best settings (hyperparameters) for our Random Forest.

In [5]:
from sklearn.model_selection import cross_val_score, GridSearchCV
import numpy as np

# 5. Model Training
pipeline.fit(X_train, y_train)
print('Model training on training set complete.')

# 6. Cross-Validation (using TimeSeriesSplit or standard KFold)
# Since it's time-series, we usually use TimeSeriesSplit, but the prompt asks for standard CV metrics.
scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-scores)
print(f'Cross-Validation RMSE: {rmse_scores.mean():.4f} (+/- {rmse_scores.std():.4f})')

# 7. Hyperparameter Tuning
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

print('\nBest Parameters found:')
print(grid_search.best_params_)
print(f'Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}')

Model training on training set complete.
Cross-Validation RMSE: 1.9402 (+/- 0.5625)

Best Parameters found:
{'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best CV RMSE: 2.2003


## 8. Best Model Selection & 9. Model Performance Evaluation

In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 8. Best Model Selection
best_model = grid_search.best_estimator_

# 9. Model Performance Evaluation
y_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('Final Model Performance on Test Set:')
print(f'RMSE: {rmse:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'R2 Score: {r2:.4f}')

Final Model Performance on Test Set:
RMSE: 2.5320
MAE:  1.8989
R2 Score: 0.7064


## 10. Web Interface with Gradio
We will create a Gradio app that takes the 5 stock prices and the 7-day moving average as input to predict the target price.

In [7]:
!pip install -q gradio
import gradio as gr

def predict_stock_price(s1, s2, s3, s4, s5, ma7):
    # Create a dataframe for the input
    input_data = pd.DataFrame([[s1, s2, s3, s4, s5, ma7]],
                             columns=['Stock_1', 'Stock_2', 'Stock_3', 'Stock_4', 'Stock_5', 'Stock_1_MA7'])
    prediction = best_model.predict(input_data)[0]
    return round(float(prediction), 2)

# Define the interface
interface = gr.Interface(
    fn=predict_stock_price,
    inputs=[
        gr.Number(label='Stock 1 Price'),
        gr.Number(label='Stock 2 Price'),
        gr.Number(label='Stock 3 Price'),
        gr.Number(label='Stock 4 Price'),
        gr.Number(label='Stock 5 Price'),
        gr.Number(label='Stock 1 (7-day MA)')
    ],
    outputs=gr.Textbox(label='Predicted Next Day Price'),
    title='Stock Price Predictor',
    description='Enter current stock prices and the 7-day moving average of Stock 1 to predict the next day's value.'
)

# Launch the app
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e35a9eeb4ecc47d2b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
